# 🗄️ Course 7 — Functions for Manipulating Data in PostgreSQL

> **Platform:** DataCamp | **Track:** Associate Data Analyst in SQL  
> **Tool:** PostgreSQL | **Database:** Sakila DVD Rental

---

## 📋 About This Course

This course covers PostgreSQL's built-in functions for working with common data types. Using the Sakila DVD Rental database, it covers data type exploration, date/time arithmetic, string manipulation, full-text search, user-defined types, and PostgreSQL extensions.

---

## 📚 Table of Contents

| Chapter | Topic |
|---------|-------|
| Chapter 1 | Overview of Common Data Types |
| Chapter 2 | Working with DATE/TIME Operators |
| Chapter 3 | Parsing and Manipulating Text |
| Chapter 4 | Full-Text Search & PostgreSQL Extensions |

---

## 📌 Chapter 1 — Overview of Common Data Types

---

### Querying INFORMATION_SCHEMA
Explore the structure of the DVD Rental database using the INFORMATION_SCHEMA system tables.

In [ ]:
-- List all public tables
SELECT * 
FROM INFORMATION_SCHEMA.TABLES
WHERE table_schema = 'public';

In [ ]:
-- List all columns in the actor table
SELECT * 
FROM INFORMATION_SCHEMA.COLUMNS
WHERE TABLE_NAME = 'actor';

### Determining Data Types
Get the column names and data types for the customer table.

In [ ]:
SELECT column_name, data_type
FROM INFORMATION_SCHEMA.COLUMNS 
WHERE table_name = 'customer';

### INTERVAL Data Type
Calculate the expected return date by adding a 3-day INTERVAL to the rental date.

In [ ]:
SELECT
	rental_date,
	return_date,
	rental_date + INTERVAL '3 DAYS' AS expected_return_date
FROM rental;

### Accessing ARRAY Data
Explore the `special_features` TEXT[] ARRAY column in the film table.

In [ ]:
-- View special_features array
SELECT title, special_features
FROM film;

In [ ]:
-- Filter by first index = 'Trailers'
SELECT title, special_features 
FROM film
WHERE special_features[1] = 'Trailers';

In [ ]:
-- Filter by second index = 'Deleted Scenes'
SELECT title, special_features 
FROM film
WHERE special_features[2] = 'Deleted Scenes';

### Searching an ARRAY with ANY
Find all films that have 'Trailers' in any position of the special_features array.

In [ ]:
SELECT title, special_features 
FROM film 
WHERE 'Trailers' = ANY(special_features);

### Searching an ARRAY with @>
Use the contains operator to find films with 'Deleted Scenes'.

In [ ]:
SELECT title, special_features 
FROM film 
WHERE special_features @> ARRAY['Deleted Scenes'];

---

## 📌 Chapter 2 — Working with DATE/TIME Operators

---

### Subtracting Dates & AGE()
Calculate the actual number of days rented using subtraction and the AGE() function.

In [ ]:
-- Method 1: Direct subtraction
SELECT f.title, f.rental_duration,
    r.return_date - r.rental_date AS days_rented
FROM film AS f
     INNER JOIN inventory AS i ON f.film_id = i.film_id
     INNER JOIN rental AS r ON i.inventory_id = r.inventory_id
ORDER BY f.title;

In [ ]:
-- Method 2: Using AGE()
SELECT f.title, f.rental_duration,
	AGE(r.return_date, r.rental_date) AS days_rented
FROM film AS f
	INNER JOIN inventory AS i ON f.film_id = i.film_id
	INNER JOIN rental AS r ON i.inventory_id = r.inventory_id
ORDER BY f.title;

### INTERVAL Arithmetic
Convert rental_duration to an INTERVAL and exclude outstanding (NULL return_date) rentals.

In [ ]:
SELECT
	f.title,
    INTERVAL '1' day * f.rental_duration,
    r.return_date - r.rental_date AS days_rented
FROM film AS f
    INNER JOIN inventory AS i ON f.film_id = i.film_id
    INNER JOIN rental AS r ON i.inventory_id = r.inventory_id
WHERE r.return_date IS NOT NULL
ORDER BY f.title;

### Calculating Expected Return Date
Add rental_duration as an INTERVAL to rental_date to get the expected return date.

In [ ]:
SELECT
    f.title, r.rental_date, f.rental_duration,
    INTERVAL '1' day * f.rental_duration + r.rental_date AS expected_return_date,
    r.return_date
FROM film AS f
    INNER JOIN inventory AS i ON f.film_id = i.film_id
    INNER JOIN rental AS r ON i.inventory_id = r.inventory_id
ORDER BY f.title;

### Current Timestamp Functions
> **Q:** Which is NOT correct?
> - A: `NOW()` → timestamp with timezone ✅
> - B: `CURRENT_TIMESTAMP` → timestamp without timezone ❌ **(wrong — it includes timezone)**
> - C: `CURRENT_DATE` → date only ✅
> - D: `CURRENT_TIME` → time only ✅

> **A: B**

In [ ]:
SELECT NOW();                          -- timestamp with timezone
SELECT CURRENT_DATE;                   -- date only
SELECT CAST(NOW() AS timestamp);       -- timestamp without timezone
SELECT NOW(), CAST(NOW() AS date);     -- both together

### Manipulating Current Date/Time
Practice using CURRENT_TIMESTAMP with precision and INTERVAL arithmetic.

In [ ]:
-- Current timestamp without timezone
SELECT CURRENT_TIMESTAMP::TIMESTAMP AS right_now;

In [ ]:
-- 5 days from now
SELECT
	CURRENT_TIMESTAMP::timestamp AS right_now,
    INTERVAL '5 DAY' + CURRENT_TIMESTAMP AS five_days_from_now;

In [ ]:
-- With 2-second precision, no fractional digits
SELECT
	CURRENT_TIMESTAMP(2)::timestamp AS right_now,
    interval '5 days' + CURRENT_TIMESTAMP(2)::timestamp AS five_days_from_now;

### EXTRACT() — Day of Week
Extract the day of week from rental_date and count rentals per day.

In [ ]:
-- Extract day of week from rental_date
SELECT EXTRACT(dow FROM rental_date) AS dayofweek 
FROM rental 
LIMIT 100;

In [ ]:
-- Count rentals by day of week
SELECT 
  EXTRACT(dow FROM rental_date) AS dayofweek, 
  COUNT(rental_date) AS rentals 
FROM rental 
GROUP BY 1;

### DATE_TRUNC() — Truncating Timestamps
Truncate rental_date by year, month, and day; then aggregate total rentals per day.

In [ ]:
SELECT DATE_TRUNC('year', RENTAL_DATE) AS rental_year FROM rental;
SELECT DATE_TRUNC('month', rental_date) AS rental_month FROM rental;
SELECT DATE_TRUNC('day', rental_date) AS rental_day FROM rental;

In [ ]:
-- Count rentals per day
SELECT 
  DATE_TRUNC('day', rental_date) AS rental_day,
  COUNT(rental_date) AS rentals
FROM rental
GROUP BY 1;

### Putting It All Together — Late Returns
Extract 90 days of rental data and flag overdue returns using EXTRACT, AGE, DATE_TRUNC, and CASE.

In [ ]:
SELECT 
  EXTRACT(dow FROM rental_date) AS dayofweek,
  AGE(return_date, rental_date) AS rental_days
FROM rental AS r 
WHERE rental_date BETWEEN CAST('2005-05-01' AS timestamp)
   AND CAST('2005-05-01' AS timestamp) + INTERVAL '90 day';

In [ ]:
SELECT 
  c.first_name || '  ' || c.last_name AS customer_name,
  f.title, r.rental_date,
  EXTRACT(dow FROM r.rental_date) AS dayofweek,
  AGE(r.return_date, r.rental_date) AS rental_days,
  CASE WHEN DATE_TRUNC('day', AGE(r.return_date, r.rental_date)) > 
    f.rental_duration * INTERVAL '1' day 
  THEN TRUE ELSE FALSE END AS past_due 
FROM film AS f 
  INNER JOIN inventory AS i ON f.film_id = i.film_id 
  INNER JOIN rental AS r ON i.inventory_id = r.inventory_id 
  INNER JOIN customer AS c ON c.customer_id = r.customer_id 
WHERE r.rental_date BETWEEN CAST('2005-05-01' AS DATE) 
  AND CAST('2005-05-01' AS DATE) + INTERVAL '90 day';

---

## 📌 Chapter 3 — Parsing and Manipulating Text

---

### Concatenating Strings
Build a formatted email 'To' field: `Brian Piccolo <bpiccolo@datacamp.com>`

In [ ]:
-- Using || operator
SELECT first_name || ' ' || last_name || ' <' || email || '>' AS full_email 
FROM customer;

In [ ]:
-- Using CONCAT()
SELECT CONCAT(first_name, ' ', last_name, ' <', email, '>') AS full_email 
FROM customer;

### Changing Case
Build a film_category field using UPPER, INITCAP, and LOWER.

In [ ]:
SELECT 
  UPPER(c.name) || ': ' || INITCAP(f.title) AS film_category, 
  LOWER(f.description) AS description
FROM film AS f 
  INNER JOIN film_category AS fc ON f.film_id = fc.film_id 
  INNER JOIN category AS c ON fc.category_id = c.category_id;

### REPLACE() — Replacing Whitespace
Replace all spaces in film titles with underscores.

In [ ]:
SELECT REPLACE(title, ' ', '_') AS title
FROM film;

### LENGTH() — String Length
Find the length of the description column.

In [ ]:
SELECT title, description,
  LENGTH(description) AS desc_len
FROM film;

### LEFT() — Truncating Strings
Get the first 50 characters of the description column.

In [ ]:
SELECT LEFT(description, 50) AS short_desc
FROM film AS f;

### SUBSTRING() + POSITION() — Extracting Street Names
Extract only the street name (without the number) from the address column.

In [ ]:
SELECT 
  SUBSTRING(address FROM POSITION(' ' IN address)+1 FOR LENGTH(address))
FROM address;

### Parsing Email — Username and Domain
Split the email column into username and domain using LEFT, SUBSTRING, and POSITION.

In [ ]:
SELECT
  LEFT(email, POSITION('@' IN email)-1) AS username,
  SUBSTRING(email FROM POSITION('@' IN email)+1 FOR LENGTH(email)) AS domain
FROM customer;

### RPAD() / LPAD() — Padding Strings
Use padding functions to concatenate first and last names.

In [ ]:
-- RPAD: add space after first_name
SELECT RPAD(first_name, LENGTH(first_name)+1) || last_name AS full_name
FROM customer;

In [ ]:
-- LPAD: add space before last_name
SELECT first_name || LPAD(last_name, LENGTH(last_name)+1) AS full_name
FROM customer;

In [ ]:
-- Full email with padding
SELECT 
	RPAD(first_name, LENGTH(first_name)+1) 
    || RPAD(last_name, LENGTH(last_name)+2, ' <') 
    || RPAD(email, LENGTH(email)+1, '>') AS full_email
FROM customer;

### TRIM() — Removing Whitespace
Truncate description to 50 chars and trim any trailing whitespace.

In [ ]:
SELECT 
  CONCAT(UPPER(c.name), ': ', f.title) AS film_category, 
  TRIM(LEFT(description, 50)) AS film_desc
FROM film AS f 
  INNER JOIN film_category AS fc ON f.film_id = fc.film_id 
  INNER JOIN category AS c ON fc.category_id = c.category_id;

### REVERSE() — Smart Truncation
Truncate description to 50 chars without cutting off a word, using REVERSE to find the last space.

In [ ]:
SELECT 
  UPPER(c.name) || ': ' || f.title AS film_category, 
  LEFT(description, 50 - 
    POSITION(' ' IN REVERSE(LEFT(description, 50)))
  ) 
FROM film AS f 
  INNER JOIN film_category AS fc ON f.film_id = fc.film_id 
  INNER JOIN category AS c ON fc.category_id = c.category_id;

---

## 📌 Chapter 4 — Full-Text Search & PostgreSQL Extensions

---

### LIKE Operator Review
Filter films using the % wildcard for different patterns.

In [ ]:
SELECT film.* FROM film WHERE title LIKE 'GOLD%';   -- starts with GOLD
SELECT * FROM film WHERE title LIKE '%GOLD';         -- ends with GOLD
SELECT * FROM film WHERE title LIKE '%GOLD%';        -- contains GOLD

### to_tsvector() — What is a tsvector?
Convert the film description to a tsvector to understand how full-text search tokenizes text.

In [ ]:
SELECT to_tsvector(description)
FROM film;

### Basic Full-Text Search
Search for the word 'elf' in film titles using to_tsvector and to_tsquery.

In [ ]:
SELECT title, description
FROM film
WHERE to_tsvector(title) @@ to_tsquery('elf');

### CREATE TYPE — User-Defined ENUM
Create a compass_position ENUM and verify it in the pg_type system table.

In [ ]:
CREATE TYPE compass_position AS ENUM (
  'North', 'South', 'East', 'West'
);

-- Verify in system table
SELECT *
FROM pg_type
WHERE typname = 'compass_position';

### Inspecting User-Defined Types
Explore the mpaa_rating ENUM type used in the film table.

In [ ]:
-- Find column info in INFORMATION_SCHEMA
SELECT column_name, data_type, udt_name
FROM INFORMATION_SCHEMA.COLUMNS 
WHERE table_name = 'film' AND column_name = 'rating';

In [ ]:
-- Inspect the type in pg_type
SELECT *
FROM pg_type
WHERE typname = 'mpaa_rating';

### User-Defined Functions — inventory_held_by_customer()
Find which films are currently rented out to customers.

In [ ]:
SELECT 
	f.title, i.inventory_id,
	inventory_held_by_customer(i.inventory_id) AS held_by_cust
FROM film AS f 
	INNER JOIN inventory AS i ON f.film_id = i.film_id
WHERE inventory_held_by_customer(i.inventory_id) IS NOT NULL;

### Enabling Extensions
Enable pg_trgm and verify both fuzzystrmatch and pg_trgm are active.

In [ ]:
CREATE EXTENSION IF NOT EXISTS pg_trgm;

-- Verify all enabled extensions
SELECT * FROM pg_extension;

### SIMILARITY() — Measuring String Similarity
Calculate the similarity between film title and description using pg_trgm.

In [ ]:
SELECT title, description, 
  SIMILARITY(title, description)
FROM film;

### levenshtein() — Fuzzy Matching
Find films matching 'JET NEIGHBOR' despite a typo using Levenshtein distance.

In [ ]:
SELECT title, description, 
  levenshtein(title, 'JET NEIGHBOR') AS distance
FROM film
ORDER BY 3;

### Putting It All Together — Full-Text + Similarity
Find films matching 'Astounding Drama' and rank them by similarity score.

In [ ]:
-- Step 1: Full-text search
SELECT title, description 
FROM film
WHERE to_tsvector(description) @@ to_tsquery('Astounding & Drama');

In [ ]:
-- Step 2: Add similarity ranking
SELECT title, description, 
  SIMILARITY(description, 'Astounding & Drama')
FROM film 
WHERE to_tsvector(description) @@ to_tsquery('Astounding & Drama') 
ORDER BY SIMILARITY(description, 'Astounding & Drama') DESC;